In [1]:
import os
import torch
from typing import Annotated, List, TypedDict, Union
from dotenv import load_dotenv

# Updated Imports
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# THE FIX: Updated text splitter import
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader

load_dotenv()

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize LLMs (Using Groq for speed and reasoning)
llm_reasoning = ChatGroq(model="llama-3.3-70b-versatile")
llm_fast = ChatGroq(model="llama-3.1-8b-instant")

# 1. Initialize Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': device}
)

# 2. Connect to your existing Indian Constitution Vector Database
db_path = "C:/Users/user/Desktop/RAG Projects/Legal RAG 1/vector_database"
constitution_vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
def create_ephemeral_store(file_path: str):
    """
    Creates a temporary in-memory vector store for the uploaded document.
    """
    # FIX: Added .lower() to handle .PDF vs .pdf and added encoding for text files
    if file_path.lower().endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    else:
        # Fallback for text files that might have special characters
        loader = TextLoader(file_path, encoding='utf-8')
        
    try:
        documents = loader.load()
    except Exception as e:
        # If UTF-8 fails for a text file, try typical Windows encoding
        if not file_path.lower().endswith('.pdf'):
            loader = TextLoader(file_path, encoding='cp1252')
            documents = loader.load()
        else:
            raise e
    
    # Recursive splitting optimized for legal formatting
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200, 
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    splits = text_splitter.split_documents(documents)
    
    # Create in-memory store (ephemeral) using the provided embeddings
    doc_vector_store = Chroma.from_documents(
        documents=splits, 
        embedding=embeddings,
        collection_name="temp_doc_analysis"
    )
    return doc_vector_store

In [3]:
# Define the Graph State
class AgentState(TypedDict):
    user_query: str
    file_path: str
    doc_store: Union[Chroma, None]
    retrieved_context: List[str]
    route: str # 'constitution', 'document', or 'comparative'
    final_answer: str

# Node 1: Router
def router_node(state: AgentState):
    query = state['user_query']
    
    prompt = f"""Analyze this legal query: "{query}"
    Determine if the user is asking about:
    1. The Indian Constitution generally (Answer: 'constitution')
    2. Specifics of the uploaded document/contract only (Answer: 'document')
    3. How the uploaded document relates to or violates the Constitution (Answer: 'comparative')
    
    Return only one word: constitution, document, or comparative."""
    
    decision = llm_fast.invoke(prompt).content.strip().lower()
    return {"route": decision}

# Updated Node 2: Context Fetcher
def context_fetcher_node(state: AgentState):
    route = state['route']
    query = state['user_query']
    context = []
    
    # Logic for Document or Comparative
    if route in ['document', 'comparative']:
        if state['doc_store']:
            # Search the uploaded document specifically
            doc_docs = state['doc_store'].similarity_search(query, k=5)
            context.append("--- UPLOADED DOCUMENT DATA ---\n" + "\n".join([d.page_content for d in doc_docs]))
        else:
            # This helps diagnose why doc tests fail
            context.append("SYSTEM ERROR: Document store is missing or not provided in state.")
            
    # Logic for Constitution or Comparative
    if route in ['constitution', 'comparative']:
        # Search the Indian Constitution database (Knowledge Base)
        kb_docs = constitution_vector_store.similarity_search(query, k=5)
        context.append("--- CONSTITUTIONAL DATA ---\n" + "\n".join([d.page_content for d in kb_docs]))
        
    return {"retrieved_context": context}

def advisor_node(state: AgentState):
    context = "\n\n".join(state['retrieved_context'])
    query = state['user_query']
    
    system_prompt = """You are a Senior Legal Advisor. 
    1. If the context contains 'UPLOADED DOCUMENT DATA', treat it as the absolute truth for the specific case.
    2. If 'CONSTITUTIONAL DATA' is present, use it for legal principles.
    3. If the context is empty or mentions 'ERROR', state that you do not have the document data.
    4. Do not hallucinate names or dates."""
    
    response = llm_reasoning.invoke([
        ("system", system_prompt),
        ("human", f"Context:\n{context}\n\nQuestion: {query}")
    ])
    return {"final_answer": response.content}

In [4]:
# Build the Graph
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("router", router_node)
workflow.add_node("fetcher", context_fetcher_node)
workflow.add_node("advisor", advisor_node)

# Set Entry Point and Edges
workflow.set_entry_point("router")
workflow.add_edge("router", "fetcher")
workflow.add_edge("fetcher", "advisor")
workflow.add_edge("advisor", END)

# Compile
legal_app = workflow.compile()

def run_legal_query(query: str, uploaded_file: str = None):
    # Initialize ephemeral store if a file is provided
    doc_store = None
    if uploaded_file and os.path.exists(uploaded_file):
        doc_store = create_ephemeral_store(uploaded_file)
        
    inputs = {
        "user_query": query,
        "file_path": uploaded_file,
        "doc_store": doc_store,
        "retrieved_context": [],
        "route": "",
        "final_answer": ""
    }
    
    result = legal_app.invoke(inputs)
    return result["final_answer"]

In [5]:
# Focus: Fact extraction from the judgment
query_1 = "What was the final decision of the court regarding the recovery of arrears of rent?"
path1 = r"C:/Users/user/Desktop/RAG Projects/Legal RAG 1/data/Mrs_Vandana_Dhirani_vs_Mrs_Arti_Kirloskar_on_27_July_2023.PDF"
print(run_legal_query(query_1,path1))

The final decision of the court was that the plaintiff's suit for recovery of arrears of rent was dismissed. The court held that the plaintiff was not entitled to claim arrears of rent as the lease had stood terminated, and the defendant had already paid the rent for the month of March 2020 and had proposed to forgo the security amount for the months of April and May 2020.


In [6]:
# Focus: Fact extraction from the judgment
query_2 = "Based on the Constitution and Indian contract law, does the 'charity immunity' or 'force majeure' claim mentioned in this case hold up against the Right to Property or tenant rights?"
path1 = r"C:/Users/user/Desktop/RAG Projects/Legal RAG 1/data/Mrs_Vandana_Dhirani_vs_Mrs_Arti_Kirloskar_on_27_July_2023.PDF"
print(run_legal_query(query_2,path1))

Based on the provided UPLOADED DOCUMENT DATA and CONSTITUTIONAL DATA, it appears that the case revolves around a lease agreement and the issue of breach of contract. The Supreme Court's decision in the case of Mrs. Vandana Dhirani vs. Mrs. Arti Kirloskar is cited, which discusses the concept of frustration of contracts and the applicability of Section 56 of the Indian Contract Act, 1872.

Regarding the "charity immunity" or "force majeure" claim, it is not explicitly mentioned in the provided data. However, the concept of force majeure is related to the doctrine of frustration of contracts, which is discussed in the case.

In the context of the Right to Property and tenant rights, the Indian Constitution provides protection under Article 300A, which states that no person shall be deprived of their property save by authority of law. The concept of force majeure or frustration of contracts may be invoked in certain circumstances, such as unforeseen events or circumstances beyond the cont

In [7]:
# List of General Questions, Keywords to check, and Reasons
general_tests = [
    ("What are the grounds for a Writ of Habeas Corpus?", "Article 32", "Check for fundamental rights and constitutional remedies."),
    ("What is the penalty for drunk driving under the Motor Vehicles Act?", "Section 185", "Verify if the agent retrieves specific act sections."),
    ("Does the Constitution provide a right to privacy?", "Puttaswamy", "Check if the agent recognizes landmark judgments."),
    ("How many members can the President nominate to Rajya Sabha?", "12", "Testing factual accuracy from the Constitution."),
    ("Under which Article is the Finance Commission constituted?", "Article 280", "Verification of administrative constitutional provisions."),
    ("Can a person be punished for the same offence twice?", "Double Jeopardy", "Check for Article 20(2) concepts."),
    ("What is the minimum age to become the Prime Minister of India?", "25", "Basic constitutional requirement check."),
    ("Is the Right to Property still a Fundamental Right?", "300A", "Check if it knows it was moved by the 44th Amendment."),
    ("Which Article deals with the abolition of Untouchability?", "Article 17", "Testing core fundamental rights."),
    ("Can Fundamental Rights be suspended during Emergency?", "Article 359", "Testing emergency provision logic.")
]

print("--- RUNNING GENERAL KB TESTS ---\n")
for i, (q, key, reason) in enumerate(general_tests):
    # run_legal_query handles the graph invocation for general KB queries
    res = run_legal_query(q)
    status = "PASS" if key.lower() in res.lower() else "FAIL"
    
    print(f"TEST #{i+1}")
    print(f"Question: {q}")
    print(f"AI Answer: {res.strip()}")
    print(f"Evaluation: {status} (Target Key: {key})")
    print(f"Reasoning: {reason}")
    print("-" * 50)

--- RUNNING GENERAL KB TESTS ---

TEST #1
Question: What are the grounds for a Writ of Habeas Corpus?
AI Answer: According to the constitutional principles, a Writ of Habeas Corpus is a legal remedy that can be sought when a person is being detained or imprisoned unlawfully. The grounds for a Writ of Habeas Corpus typically include:

1. Unlawful detention or imprisonment: When a person is being held without a valid reason or authority.
2. Lack of due process: When a person is being detained without being given a fair trial or opportunity to defend themselves.
3. Violation of constitutional rights: When a person's constitutional rights, such as the right to liberty or the right to a fair trial, are being violated.

The specific grounds may vary depending on the jurisdiction and the circumstances of the case. However, the overarching principle is that a Writ of Habeas Corpus is a means to challenge the legality of a person's detention and to protect their fundamental rights.
Evaluation: 

In [8]:
# Path to your uploaded document
doc_path = r"C:/Users/user/Desktop/RAG Projects/Legal RAG 1/data/Mrs_Vandana_Dhirani_vs_Mrs_Arti_Kirloskar_on_27_July_2023.PDF"

# Pre-load the store ONCE so it isn't empty when the test starts
shared_doc_store = create_ephemeral_store(doc_path)

doc_tests = [
    ("Who is the plaintiff in this case?", "Vandana Dhirani", "Check entity extraction."),
    ("What is the CS DJ number of this suit?", "588/2021", "Check specific document metadata extraction."),
    ("What was the result of the suit announced on 27.07.2023?", "Dismissed", "Check case outcome accuracy."),
    ("What was Issue No. 8 about?", "Interest", "Check specific legal issue identification."),
    ("Who was the presiding judge in the Saket Courts for this case?", "Navjeet Budhiraja", "Check judge name extraction."),
    ("Did the court allow the recovery of arrears of rent?", "not found entitled", "Check core judgment findings."),
    ("At what floor is the property C-99 located?", "4th Floor", "Check property detail extraction."),
    ("What was the date of institution of this suit?", "31.08.2021", "Check procedural dates."),
    ("What reason did the defendant give for not paying rent (Issue 4)?", "force majeure", "Check for defense arguments (Covid/Force Majeure)."),
    ("Was a decree sheet ordered to be prepared?", "Decree sheet be prepared", "Check final court directions.")
]

print("\n--- RUNNING DOCUMENT-SPECIFIC TESTS ---\n")
for i, (q, key, reason) in enumerate(doc_tests):
    # Manually preparing inputs to ensure shared_doc_store is used
    inputs = {
        "user_query": q,
        "file_path": doc_path,
        "doc_store": shared_doc_store, 
        "retrieved_context": [],
        "route": "",
        "final_answer": ""
    }
    
    result = legal_app.invoke(inputs)
    res = result["final_answer"]
    
    # INITIAL PASS CHECK
    status = "PASS" if key.lower() in res.lower() else "FAIL"
    
    # FLEXIBLE PASS CHECK (Handling synonyms or common legal phrasing)
    if status == "FAIL":
        if "dismiss" in res.lower() and key.lower() == "dismissed":
            status = "PASS (Synonym)"
        elif "not entitled" in res.lower() and key.lower() == "not found entitled":
            status = "PASS (Synonym)"
        elif "saket" in res.lower() and key.lower() == "navjeet budhiraja":
            # If it found the court but missed the judge name initially
            status = "FAIL (Partial Match)"
    
    print(f"DOC TEST #{i+1}")
    print(f"Question: {q}")
    print(f"AI Answer: {res.strip()}")
    print(f"Evaluation: {status} (Target Key: {key})")
    print(f"Reasoning: {reason}")
    print("-" * 60)


--- RUNNING DOCUMENT-SPECIFIC TESTS ---

DOC TEST #1
Question: Who is the plaintiff in this case?
AI Answer: I do not have the document data.
Evaluation: FAIL (Target Key: Vandana Dhirani)
Reasoning: Check entity extraction.
------------------------------------------------------------
DOC TEST #2
Question: What is the CS DJ number of this suit?
AI Answer: I do not have the document data.
Evaluation: FAIL (Target Key: 588/2021)
Reasoning: Check specific document metadata extraction.
------------------------------------------------------------
DOC TEST #3
Question: What was the result of the suit announced on 27.07.2023?
AI Answer: I do not have the document data.
Evaluation: FAIL (Target Key: Dismissed)
Reasoning: Check case outcome accuracy.
------------------------------------------------------------
DOC TEST #4
Question: What was Issue No. 8 about?
AI Answer: I do not have the document data.
Evaluation: FAIL (Target Key: Interest)
Reasoning: Check specific legal issue identification